# Atividade Prática: Desenvolvendo um Sistema de RH com SQLAlchemy

## Nível 1: Básico — Configuração e SQL Puro com Segurança

In [ ]:
import os
import pandas as pd
from sqlalchemy import (Column, Float, Integer, MetaData, String, Table,
                        and_, or_, create_engine, delete, func, insert,
                        select, text, update)

In [ ]:
# Reinicia o banco: remove o arquivo anterior, se existir
if os.path.exists('loja.db'):
    os.remove('loja.db')

Passo 1: Crie a conexão com um banco de dados local utilizando a função create_engine('sqlite:///sistema_rh.db').

In [ ]:
# SQLite em arquivo — sem necessidade de servidor
engine = create_engine('sqlite:///sistema_rh.db')

print('Engine pronto. Banco:', engine.url.database)

Engine pronto. Banco: sistema_rh.db


Passo 2: Abra uma transação com with engine.begin() as conn: e utilize a função text() para executar um comando CREATE TABLE em SQL puro, criando uma tabela chamada funcionarios (com id, nome, cargo e salario).

In [ ]:
with engine.begin() as conn:
  conn.execute(text('''
      CREATE TABLE funcionario (
          id INTEGER PRIMARY KEY,
          nome TEXT,
          cargo TEXT,
          salaro REAL
      )
  '''))

print('Tabela "funcionario" criada!')

Passo 3 (Segurança): Simule a inserção de um novo funcionário a partir de um formulário web. Utilize o comando INSERT passando parâmetros seguros (ex: :nome, :cargo) e um dicionário de valores correspondentes. Pergunta reflexiva para os alunos: Por que nunca devemos concatenar strings diretamente no SQL (risco de injeção SQL) e como os placeholders resolvem isso?


In [ ]:
with engine.begin() as conn:
    conn.execute(text('''
        INSERT INTO funcionario (id, nome, cargo, salaro)
        VALUES (:id, :nome, :cargo, :salaro)
    '''), [
        {'id': 1, 'nome': 'Paulo', 'cargo': 'Analista', 'salaro': 5000.00}
    ])

print('1 registro inserido.')

1 registro inserido


Passo 4: Valide a inserção consultando os dados com a função pd.read_sql_query(), que já retorna a tabela formatada diretamente como um DataFrame do Pandas.

In [ ]:
#---Forma 1
# with engine.connect() as conn:
#     resultado = conn.execute(text('SELECT * FROM funcionario'))
#     linhas = resultado.mappings().all()

# print('linhas:', linhas)

# df_funcionarios = pd.DataFrame(linhas)
# df_funcionarios

#---Forma 2
df_funcionarios = pd.read_sql_query('SELECT * FROM funcionario', engine)
df_funcionarios

,id,nome,cargo,salaro
0,1,Paulo,Analista,5000.0


## Nível 2: Intermediário — SQLAlchemy Core (Automatização Programática)

Passo 1: Defina uma nova tabela chamada projetos de forma programática utilizando os objetos Table, MetaData e Column. Em seguida, crie a tabela fisicamente no banco com metadata.create_all(engine).

In [ ]:
metadata = MetaData()

tabela_projeto = Table(
    'projeto', metadata,
    Column('id', Integer, primary_key=True),
    Column('nome', String(60)),
)

metadata.create_all(engine)
print('Tabela projeto criada')


Tabela projeto criada


Passo 2: Receba uma lista de dicionários contendo múltiplos projetos e faça uma inserção em lote (bulk insert) passando essa lista para o comando conn.execute(insert(projetos), [lista_de_dicts]).

In [ ]:
registros = [
    {'nome': 'Projeto A'},
    {'nome': 'Projeto B'},
    {'nome': 'Projeto C'},
]

with engine.begin() as conn:
    conn.execute(insert(projeto), registros)

print('3 projetos inseridos')

3 projetos inseridos


Passo 3: A diretoria aprovou um reajuste. Utilize a instrução update(tabela).where(...).values(...) para aumentar o salário apenas dos funcionários que ocupam o cargo de 'Desenvolvedor Júnior'.


In [ ]:
tabela_funcionario = Table(
    'funcionario', metadata,
    Column('id', Integer, primary_key=True),
    Column('nome', String),
    Column('cargo', String),
    Column('salaro', Float),
    extend_existing=True
)

with engine.begin() as conn:
    conn.execute(text('''
        INSERT OR IGNORE INTO funcionario (id, nome, cargo, salaro)
        VALUES (:id, :nome, :cargo, :salaro)
    '''), [
        {'id': 2, 'nome': 'Carla', 'cargo': 'Desenvolvedor Júnior', 'salaro': 2500.00}
    ])

    conn.execute(
        update(tabela_funcionario)
        .where(tabela_funcionario.c.cargo == 'Desenvolvedor Júnior')
        .values(salaro=tabela_funcionario.c.salaro + 500)
    )

pd.read_sql_query('SELECT * FROM funcionario', engine)

,id,nome,cargo,salaro
0,1,Paulo,Analista,5000.0
1,2,Carla,Desenvolvedor Júnior,3000.0


Passo 4: Gere um relatório salarial agregando os dados. Construa um select combinando func.avg() para calcular a média salarial e group_by() para agrupar o resultado por cargo.

In [ ]:
stmt = (select(tabela_funcionario.c.cargo, func.avg(tabela_funcionario.c.salaro).label('media_salarial'))
        .group_by(tabela_funcionario.c.cargo))

pd.read_sql_query(stmt, engine)

,cargo,media_salarial
0,Analista,5000.0
1,Desenvolvedor Júnior,3000.0


## Nível 3: Avançado — ORM (Orientação a Objetos e Relacionamentos)


Passo 1: Transforme as tabelas em classes Python. Utilize declarative_base() e defina as classes Departamento e FuncionarioORM mapeando as colunas com mapped_column.

---

Passo 2: Estabeleça a relação entre as classes configurando uma ForeignKey na tabela de funcionários e utilizando a função relationship em ambas as classes para permitir a navegação de objeto para objeto.

In [ ]:
from sqlalchemy import ForeignKey
from sqlalchemy.orm import declarative_base, mapped_column, relationship, sessionmaker

In [ ]:
Base = declarative_base()

class Departamento(Base):
    """Um departamento pode ter vários funcionários."""
    __tablename__ = 'departamento_orm'

    id = mapped_column(Integer, primary_key=True)
    nome = mapped_column(String(60))
    uf = mapped_column(String(2))

    funcionarios = relationship('Funcionario', back_populates='departamento')


class Funcionario(Base):
    """Um funcionário trabalha em um departamento."""
    __tablename__ = 'funcionario_orm'

    id = mapped_column(Integer, primary_key=True)
    nome = mapped_column(String(60))
    departamento_id = mapped_column(ForeignKey('departamento_orm.id'))

    departamento = relationship('Departamento', back_populates='funcionarios')


Base.metadata.create_all(engine)
print('Tabelas ORM criadas:', list(Base.metadata.tables.keys()))

Tabelas ORM criadas: ['departamento_orm', 'funcionario_orm']


Passo 3: Crie uma fábrica de sessões com sessionmaker(bind=engine) e instancie uma Session. Crie um objeto da classe Departamento e adicione funcionários a ele. Persista todos no banco de dados de uma só vez utilizando sessao.add() e sessao.commit().

In [ ]:
Session = sessionmaker(bind=engine)
sessao = Session()

depTI = Departamento(id=1, nome='Tecnologia', uf='RJ')
paulo = Funcionario(id=1, nome='Paulo Pinheiro', departamento_id=1)
isabelle = Funcionario(id=2, nome='Isabelle Pinheiro', departamento_id=1)

depTI.funcionarios = [paulo, isabelle]

sessao.add_all([paulo, isabelle])
sessao.commit()

print(sessao.query(Departamento).all())
print(sessao.query(Funcionario).all())

[<__main__.Funcionario object at 0x7d15369d16a0>, <__main__.Funcionario object at 0x7d1537048a50>]


Passo 4: Faça uma consulta orientada a objetos para listar todos os funcionários do departamento de "TI". Utilize sessao.execute(select(...)).scalars().all() para que o SQLAlchemy retorne os objetos instanciados da classe em vez de linhas textuais. No final, utilize sessao.close() para fechar a sessão adequadamente.


In [ ]:
func_ti = sessao.execute(select(Funcionario).where(Funcionario.departamento_id == 1)).scalars().all()
print('Funcionários de TI:', [f.nome for f in func_ti])

Funcionários de TI: ['Paulo Pinheiro', 'Isabelle Pinheiro']
